In [47]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

LEVELS = range(10)

def compute_multi_level_ofi(df):
    multi_level_ofi = []

    for i in range(1, len(df)):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]

        level_ofis = []
        for lvl in LEVELS:
            bid_px = f'bid_px_0{lvl}'
            bid_sz = f'bid_sz_0{lvl}'
            ask_px = f'ask_px_0{lvl}'
            ask_sz = f'ask_sz_0{lvl}'

            if curr[bid_px] > prev[bid_px]:
                bid_ofi = curr[bid_sz]
            elif curr[bid_px] == prev[bid_px]:
                bid_ofi = curr[bid_sz] - prev[bid_sz]
            else:
                bid_ofi = -prev[bid_sz]

            if curr[ask_px] < prev[ask_px]:
                ask_ofi = curr[ask_sz]
            elif curr[ask_px] == prev[ask_px]:
                ask_ofi = prev[ask_sz] - curr[ask_sz]
            else:
                ask_ofi = -prev[ask_sz]

            level_ofis.append(bid_ofi - ask_ofi)

        multi_level_ofi.append({
            'timestamp': curr['ts_event'],
            'symbol': curr['symbol'],
            'best_level_ofi': level_ofis[0],
            **{f'ofi_level_{lvl}': level_ofis[lvl] for lvl in LEVELS}
        })

    return pd.DataFrame(multi_level_ofi)


def compute_integrated_ofi(multi_level_df):
    ofi_cols = [f'ofi_level_{lvl}' for lvl in LEVELS]
    X = multi_level_df[ofi_cols]
    X_scaled = StandardScaler().fit_transform(X)
    integrated = PCA(n_components=1).fit_transform(X_scaled)
    multi_level_df['integrated_ofi'] = integrated.flatten()
    return multi_level_df

def compute_cross_asset_ofi(integrated_df):
    cross_asset_values = []

    for ts, group in integrated_df.groupby('timestamp'):
        cross_map = {
            row['symbol']: group[group['symbol'] != row['symbol']]['integrated_ofi'].sum()
            for _, row in group.iterrows()
        }

        for _, row in group.iterrows():
            cross_asset_values.append(cross_map[row['symbol']])

    integrated_df['cross_asset_ofi'] = cross_asset_values
    return integrated_df




df = pd.read_csv('data/first_25000_rows.csv')
multi_level_df = compute_multi_level_ofi(df)
multi_level_df = compute_integrated_ofi(multi_level_df)
final_df = compute_cross_asset_ofi(multi_level_df)
final_df

,timestamp,symbol,best_level_ofi,ofi_level_0,ofi_level_1,ofi_level_2,ofi_level_3,ofi_level_4,ofi_level_5,ofi_level_6,ofi_level_7,ofi_level_8,ofi_level_9,integrated_ofi,cross_asset_ofi
0,2024-10-21T11:54:29.223769812Z,AAPL,2,2,0,0,0,0,0,0,0,0,0,0.006811,0.0
1,2024-10-21T11:54:29.225030400Z,AAPL,3,3,0,0,0,0,0,0,0,0,0,0.009807,0.0
2,2024-10-21T11:54:29.712434212Z,AAPL,0,0,0,200,0,0,0,0,0,0,0,0.199861,0.0
3,2024-10-21T11:54:29.764673165Z,AAPL,0,0,0,-200,0,0,0,0,0,0,0,-0.198220,0.0
4,2024-10-21T11:54:29.764685027Z,AAPL,0,0,400,10,0,0,0,0,0,0,0,0.508293,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4994,2024-10-21T13:04:16.583527688Z,AAPL,0,0,0,200,15,109,10,45,110,100,55,0.774523,0.0
4995,2024-10-21T13:04:17.976461017Z,AAPL,0,0,200,0,0,0,0,0,0,0,0,0.249581,0.0
4996,2024-10-21T13:04:20.085638629Z,AAPL,0,0,0,-200,-15,-109,-10,-45,-110,-100,-55,-0.772882,0.0
4997,2024-10-21T13:04:20.085651109Z,AAPL,0,0,0,0,200,109,10,45,110,100,55,0.763888,0.0


In [48]:
user_given_timestamp = str(input())


if(user_given_timestamp==df['ts_event'].iloc[0]):
    print("Not possible to have OFI data for the first timestamp")
else: 
    result = final_df.loc[final_df.timestamp == user_given_timestamp]
result

,timestamp,symbol,best_level_ofi,ofi_level_0,ofi_level_1,ofi_level_2,ofi_level_3,ofi_level_4,ofi_level_5,ofi_level_6,ofi_level_7,ofi_level_8,ofi_level_9,integrated_ofi,cross_asset_ofi
0,2024-10-21T11:54:29.223769812Z,AAPL,2,2,0,0,0,0,0,0,0,0,0,0.006811,0.0
